# 01 · Algonauts data preparation

This notebook handles **only the one-time acquisition, Drive backup, and session restore** of the data required by the BEATs-to-fMRI encoding pipeline.

It fetches only:
- one Friends / Schaefer-1000 fMRI `.h5` file for each configured subject;
- all available video parts (`a`, `b`, …) for Season 2 Episodes 1–5.

It does **not** fetch atlas volumes or prepare hierarchy/spatial-map resources. Those belong to a later, separate analysis step.

**Run order:** mount Drive → (optionally) acquire and back up the dataset once → restore the backup to local disk whenever a new Colab session starts.


## 1. Mount Google Drive and locate the project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# Change this only if you extracted the project folder somewhere else in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/BEATs_algonauts_project')
if not (PROJECT_DIR / 'config.py').exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}. Update PROJECT_DIR above.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['BEATS_BRAIN_IN_COLAB'] = '1'  # inherited by Python subprocesses
os.environ['BEATS_BRAIN_DRIVE_ROOT'] = str(PROJECT_DIR)
os.environ['BEATS_BRAIN_WORK_DIR'] = str(PROJECT_DIR / 'outputs')

import config
import get_algonauts_data as algonauts

print('Project:', PROJECT_DIR)
print('Permanent data backup:', config.DRIVE_DATA)
print('Local working data:', config.ALGONAUTS_ROOT)


Mounted at /content/drive
Project: /content/drive/MyDrive/BEATs_algonauts_project
Permanent data backup: /content/drive/MyDrive/algonauts_2025
Local working data: /content/data


## 2. Download the required files and back them up to Drive

Set `RUN_DATA_DOWNLOAD = True` only when the permanent Drive backup does not yet contain the required files (or you intentionally need to fetch them again).

DataLad installs its metadata clone on local disk under `/content`, not inside Google Drive. Only the paths selected by `get_algonauts_data.py` are fetched.


In [3]:
RUN_DATA_DOWNLOAD = True  # Set True only for the initial download / an intentional re-fetch.

if RUN_DATA_DOWNLOAD:
    repo, selection = algonauts.fetch_all(parent_dir='/content', skip_tools=False, dry_run=False)
    print('\nSelected files:')
    print(algonauts.describe(selection, repo))

    present, total = algonauts.fetch_report(selection)
    if present != total:
        raise RuntimeError(f'Only {present}/{total} selected files are present after DataLad fetch.')

    copied, skipped, missing = algonauts.backup_to_drive(selection, repo, config.DRIVE_DATA)
    print(f'\nDrive backup complete: copied={copied}, already present={skipped}, missing={missing}')
else:
    print('Skipping DataLad download. The notebook will use the existing Drive backup.')


$ /usr/bin/python3 -m pip -q install datalad datalad-installer
$ datalad-installer --sudo ok --log-level DEBUG git-annex -m datalad/git-annex:release
Using existing DataLad installation: /content/algonauts_2025.competitors
fMRI files: 4 | video clips: 10
  - fmri/sub-01/func/sub-01_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5
  - fmri/sub-02/func/sub-02_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5
  - fmri/sub-03/func/sub-03_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5
  - fmri/sub-05/func/sub-05_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5
  - stimuli/movies/friends/s2/friends_s02e01a.mkv
  - stimuli/movies/friends/s2/friends_s02e01b.mkv
  - stimuli/movies/friends/s2/friends_s02e02a.mkv
  - stimuli/movies/friends/s2/friends_s02e02b.mkv
  - stimuli/movies/friends/s2/friends_s02e03a.mkv

## 3. Restore the permanent backup to fast local disk

In [4]:
if config.DRIVE_DATA is None:
    raise RuntimeError('DRIVE_DATA is not configured. Check config.py / ALGONAUTS_DRIVE_DATA.')

copied, skipped = algonauts.restore_from_drive(config.DRIVE_DATA, config.ALGONAUTS_ROOT)
print(f'Restored dataset files: copied={copied}, already present={skipped}')


Restored dataset files: copied=14, already present=0


## 4. Verify the files needed by the encoding pipeline

In [5]:
import re

missing_fmri = []
for subject in config.SUBJECTS:
    hits = sorted((config.FMRI_DIR / subject / 'func').glob(config.FMRI_GLOB))
    print(f'{subject}: {len(hits)} matching fMRI file(s)')
    if len(hits) != 1:
        missing_fmri.append((subject, [p.name for p in hits]))

stimulus_dir = config.STIM_DIR / f's{config.SEASON}'
clip_pattern = re.compile(
    rf's0?-?{config.SEASON}e0?({'|'.join(map(str, sorted(set(config.TRAIN_EPS) | {config.TEST_EP})))})[a-z]\.mkv$',
    re.IGNORECASE,
)
clips = sorted(p for p in stimulus_dir.glob('*.mkv') if clip_pattern.search(p.name))
print(f'Video clips found: {len(clips)}')
for clip in clips:
    print(' -', clip.name)

if missing_fmri or not clips:
    raise FileNotFoundError(f'Data verification failed. fMRI issues: {missing_fmri}; video directory: {stimulus_dir}')

found_episodes = {
    int(re.search(r'e0?(\d+)[a-z]\.mkv$', p.name, re.IGNORECASE).group(1)) for p in clips
}
required_episodes = set(config.TRAIN_EPS) | {config.TEST_EP}
if not required_episodes.issubset(found_episodes):
    raise FileNotFoundError(f'Missing episode clips: {sorted(required_episodes - found_episodes)}')

print('\nData preparation is complete. The main pipeline can now use config.ALGONAUTS_ROOT.')


sub-01: 1 matching fMRI file(s)
sub-02: 1 matching fMRI file(s)
sub-03: 1 matching fMRI file(s)
sub-05: 1 matching fMRI file(s)
Video clips found: 10
 - friends_s02e01a.mkv
 - friends_s02e01b.mkv
 - friends_s02e02a.mkv
 - friends_s02e02b.mkv
 - friends_s02e03a.mkv
 - friends_s02e03b.mkv
 - friends_s02e04a.mkv
 - friends_s02e04b.mkv
 - friends_s02e05a.mkv
 - friends_s02e05b.mkv

Data preparation is complete. The main pipeline can now use config.ALGONAUTS_ROOT.
